# CAPTCHA Classifier — Retrain YOLOv8 model

The goal of this notebook is to let you retrain the existing classifier based on a new captcha type that you have created a solution for.

This notebook performs the following steps:
1. The first cell creates synthetic captchas from a recaptchav2 dataset, these are used for the no solver class
2. In cell 1, all the paths to the datasets are initialized. There you can add your new class with the correct data path
3. NOTE: IF YOU HAVE CREATED A SOLUTION FOR A CAPTCHA TYPE THAT IS CURRENTLY LINKED TO THE NO SOLVER CLASS, REMOVE IT FROM THE NO SOLVER CLASS

   - (e.g.) if you have created a solution to the scrambled image puzzle, then remove the scrambled_image_v1 and v2 datasets from the no solver class data paths in cell 1 

4. After running all cells, the new classification model will be saved to the OUTPUT_DIR. Then you can manually swap it for the existing models/CLASSIFICATION_MODEL.pt file if you want

## 0. (Optional) Generate fake captchas for the *No Solver* class

Only needed for first time running. Afterwards, the images already exist in the data/classifier/fake_no_solver/ directory.

Note that it creates recaptchav2 and odd-one-out style images. If you create a solution for any of these two classes, remove these data paths from the no solver category in cell 1

In [ ]:
import random
from pathlib import Path
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

# ------------------------------- config -------------------------------
INPUT_DIR = Path("data/classifier/recaptchav2")      # source pool for fakes
OUT_BASE  = Path("data/classifier/fake_no_solver")   # written here, per category
N_3X3, N_4X4, N_ODD = 1000, 1000, 1000
PREVIEW_ONLY = False
SEED        = 42
TILE, GAP   = 100, 6
WHITE, BLACK = (255, 255, 255), (0, 0, 0)
IMG_EXTS    = (".png", ".jpg", ".jpeg", ".bmp", ".webp")
# ----------------------------------------------------------------------
rng = random.Random(SEED)

def _list_images(root):
    return [p for p in Path(root).rglob("*")
            if p.is_file() and p.suffix.lower() in IMG_EXTS
            and not any(s.startswith(".") for s in p.parts)]

pool = _list_images(INPUT_DIR)
assert len(pool) >= 9, f"need >=9 source images, found {len(pool)} in {INPUT_DIR}"
print(f"{len(pool)} source images found")

def _load(p):  return Image.open(p).convert("RGB")

def _assemble(tiles, rows, cols, bg):
    W = cols * TILE + (cols + 1) * GAP
    H = rows * TILE + (rows + 1) * GAP
    canvas = Image.new("RGB", (W, H), bg)
    for i, t in enumerate(tiles):
        r, c = divmod(i, cols)
        canvas.paste(t.resize((TILE, TILE)), (GAP + c * (TILE + GAP), GAP + r * (TILE + GAP)))
    return canvas

def _slice_grid(img, n):
    big = img.resize((n * TILE, n * TILE))
    return [big.crop((c * TILE, r * TILE, (c + 1) * TILE, (r + 1) * TILE))
            for r in range(n) for c in range(n)]

def make_3x3():        return _assemble([_load(p) for p in rng.sample(pool, 9)], 3, 3, WHITE)
def make_4x4_single(): return _assemble(_slice_grid(_load(rng.choice(pool)), 4), 4, 4, WHITE)
def make_odd_one_out():
    base  = rng.choice(pool)
    tiles = _slice_grid(_load(base), 4)
    for idx in rng.sample(range(16), rng.randint(3, 5)):
        op = rng.choice(pool)
        while op == base:
            op = rng.choice(pool)
        tiles[idx] = _load(op).resize((TILE, TILE))
    return _assemble(tiles, 4, 4, BLACK)

specs = [("recaptcha_3x3", N_3X3, make_3x3),
         ("recaptcha_4x4", N_4X4, make_4x4_single),
         ("odd_one_out",   N_ODD, make_odd_one_out)]

if PREVIEW_ONLY:
    print("PREVIEW_ONLY=True -> set False to generate.")
else:
    for name, n, fn in specs:
        d = OUT_BASE / name; d.mkdir(parents=True, exist_ok=True)
        for i in range(n):
            fn().save(d / f"{name}_{i:05d}.png")
        print(f"wrote {n} -> {d}")

## 1. Config — **add your new dataset(s) and class(es) here**

In [ ]:
import os, random, hashlib, re, shutil, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
from PIL import Image, ImageFile, ImageDraw
ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings("ignore")

# Run the notebook from the repo root (or make these absolute).
DATA_DIR   = Path("data")
OUTPUT_DIR = Path("classifier_output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# THESE FILES REQUIRE THE DATA STRUCTURE OF THE data FOLDER AS SPECIFIED IN THE README_RETRAIN_CLASSIFIER.md FILE
# IF YOUR FILE PATHS ARE DIFFERENT, MAKE SURE TO CHANGE THEM HERE
PATHS = {
    "gregwar":      DATA_DIR / "text" / "Gregwar", # contains images directly
    "mobicms":      DATA_DIR / "text" / "Mobicms", # contains images directly
    "king":         DATA_DIR / "text" / "King", # contains images directly
    "text_other":  [DATA_DIR / "text" / "Wang"], # contains folders of different schemes, each with test, val, train
    "moving_train": DATA_DIR / "moving_window" / "addyrus_images", # contains images directly
    "moving_test":  DATA_DIR / "moving_window" / "darkmarketreloaded_images", # contains images directly
    "open_circle":  DATA_DIR / "open_circle" / "images", # contains images directly
    "rot_normal":  [DATA_DIR / "image_rotation" / "default" / "Pitch", # contains images directly
                    DATA_DIR / "image_rotation" / "default" / "Dread"], # contains images directly
    "rot_special":  DATA_DIR / "image_rotation" / "special", # contains images directly, that will be transformed in a later cell
    # REMOVE THE DATA PATHS BELOW FOR THE ONES THAT YOU CREATE A SOLUTION FOR, IF ANY
    "no_solver":   [
        DATA_DIR / "classifier" / "odd_one_out_v1", # contains images directly
        DATA_DIR / "classifier" / "security_question_v1", # contains images directly
        DATA_DIR / "classifier" / "security_question_v2", # contains images directly
        DATA_DIR / "classifier" / "scrambled_image_v1", # contains images directly
        DATA_DIR / "classifier" / "scrambled_image_v2", # contains images directly
        DATA_DIR / "classifier" / "math", # contains images directly
        DATA_DIR / "classifier" / "minority_group", # contains images directly
        DATA_DIR / "classifier" / "fake_no_solver" / "recaptcha_3x3", # contains images directly, created by previous cell
        DATA_DIR / "classifier" / "fake_no_solver" / "recaptcha_4x4", # contains images directly, created by previous cell
        DATA_DIR / "classifier" / "fake_no_solver" / "odd_one_out", # contains images directly, created by previous cell
    ],
}

# THESE ARE THE EXISTING CLASSES, DONT CHANGE THEM. BELOW YOU CAN ADD YOUR NEW CLASSES
BASE_CLASSES = [
    "Text (Gregwar)", "Text (Mobicms)", "Text (King)", "Text (other)", "Moving Window",
    "Open Circle", "Image Rotation (normal)", "Image Rotation (special)", "No Solver",
]
BASE_SAFE = {
    "Text (Gregwar)": "text_gregwar", "Text (Mobicms)": "text_mobicms",
    "Text (King)": "text_king", "Text (other)": "text_other",
    "Moving Window": "moving_window", "Open Circle": "open_circle",
    "Image Rotation (normal)": "img_rotation_normal",
    "Image Rotation (special)": "img_rotation_special", "No Solver": "no_solver",
}

# ============================================================================
# >>> ADD YOUR NEW CAPTCHA TYPE(S) HERE <<<
#
# FIRST MAKE SURE TO ADD THE DATASET TO THE data/ FOLDER
# USE THE EXISTING FORMAT FOR THAT: data/image_puzzle_name/dataset_A
# YOU MAY USE MULTIPLE DATASETS UNDER THE DIRECTORY, AS LONG AS EVERYTHING INSIDE THAT DATASET ARE IMAGES
#
# One tuple per new type:
#     ("Readable class name", "safe_dir_name", ["/path/to/images", ...])
#   * Readable name : the class label (keep it human-readable)
#   * safe_dir_name : lowercase, no spaces/punctuation; becomes the folder name
#                     AND the YOLO label. Use this SAME string in DACCAS's
#                     SAFE_TO_CLASS map (daccas/classifier.py).
#   * paths         : one or more folders of images for this new type
#
# Leave the list empty ([]) to just retrain on the existing classes.
# ============================================================================
NEW_CLASSES = [
    # ("Slider Puzzle", "slider_puzzle", [
    #     DATA_DIR / "slider_puzzle" / "datasetA",
    #     DATA_DIR / "slider_puzzle" / "datasetB",
]),
]

# GROUP THE NEW CLASSES WITH THE EXISTING CLASSES
CLASSES = list(BASE_CLASSES) + [name for (name, _safe, _paths) in NEW_CLASSES]
SAFE    = dict(BASE_SAFE);  SAFE.update({name: safe for (name, safe, _paths) in NEW_CLASSES})
NEW_PATHS = {name: paths for (name, _safe, paths) in NEW_CLASSES}

# sanity: unique names + safe names
assert len(CLASSES) == len(set(CLASSES)), "duplicate class name in NEW_CLASSES"
assert len(set(SAFE.values())) == len(SAFE), "duplicate safe_dir_name in NEW_CLASSES"

CLASS_TO_IDX  = {c: i for i, c in enumerate(CLASSES)}
NUM_CLASSES   = len(CLASSES)
SAFE_TO_IDX   = {SAFE[c]: CLASS_TO_IDX[c] for c in CLASSES}
SAFE_TO_CLASS = {SAFE[c]: c for c in CLASSES}

WORK_DIR    = OUTPUT_DIR
STITCH_DIR  = WORK_DIR / "special_rotation_stitched"
DATASET_DIR = WORK_DIR / "cls_dataset"

# ----------------------------- knobs -----------------------------
SEED          = 42
CAP_PER_CLASS = 2500
CAP_OVERRIDES = {"No Solver": None}     # None = keep ALL for that class
SPLIT         = (0.80, 0.10, 0.10)      # train / val / test per class
COMBINE_MODE  = "overlay"               # special-rotation stitch mode

OVERSAMPLE_TRAIN       = True
TARGET_TRAIN_PER_CLASS = round(CAP_PER_CLASS * SPLIT[0])

# YOLO training — single model only (the DACCAS dispatcher is yolov8n-cls)
MODEL      = "yolov8n-cls.pt"
IMG_SIZE   = 224
EPOCHS     = 100
PATIENCE   = 10
BATCH_SIZE = 64

IMG_EXTS = (".png", ".jpg", ".jpeg", ".bmp", ".webp")

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    try:
        import torch; torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    except Exception:
        pass
seed_everything()

print(f"{NUM_CLASSES} classes:", CLASSES)
if NEW_CLASSES:
    print("New classes added:", [n[0] for n in NEW_CLASSES])

## 2. Helpers (listing, dedup, special-rotation stitching)

In [ ]:
def _visible(p):
    return not any(part.startswith(".") for part in Path(p).parts)

def list_images(root, recursive=True):
    root = Path(root)
    if not root.exists():
        return []
    if root.is_file():
        return [root] if (root.suffix.lower() in IMG_EXTS and _visible(root)) else []
    it = root.rglob("*") if recursive else root.glob("*")
    return [p for p in it if p.is_file() and p.suffix.lower() in IMG_EXTS and _visible(p)]

def dedup_by_pixels(paths):
    seen, unique = set(), []
    for p in paths:
        try:
            with Image.open(p) as im:
                h = hashlib.md5(np.asarray(im.convert("RGB")).tobytes()).hexdigest()
        except Exception:
            continue
        if h not in seen:
            seen.add(h); unique.append(p)
    return unique

def _vsplit4(img):
    w, h = img.size; step = w // 4
    return [img.crop((i * step, 0, (i + 1) * step if i < 3 else w, h)) for i in range(4)]

def _disc_mask(size, ss=4):
    w, h = size
    m = Image.new("L", (w * ss, h * ss), 0)
    ImageDraw.Draw(m).ellipse((0, 0, w * ss - 1, h * ss - 1), fill=255)
    return m.resize((w, h), Image.LANCZOS)

def _combine(ring, circ, mode):
    if mode == "horizontal":
        out = Image.new("RGB", (ring.width + circ.width, max(ring.height, circ.height)), "white")
        out.paste(ring, (0, 0)); out.paste(circ, (ring.width, 0)); return out
    if mode == "overlay":
        base = ring.convert("RGB").copy(); circ = circ.convert("RGB")
        cx = (base.width - circ.width) // 2; cy = (base.height - circ.height) // 2
        base.paste(circ, (cx, cy), _disc_mask(circ.size)); return base
    out = Image.new("RGB", (max(ring.width, circ.width), ring.height + circ.height), "white")
    out.paste(ring, (0, 0)); out.paste(circ, (0, ring.height)); return out

def build_special_rotation(src, out_dir, mode=COMBINE_MODE):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    files = list_images(src, recursive=True)
    rings, circles = {}, {}
    pat = re.compile(r"(?:^|_)?(?:it_)?(\d+)_(rings|circles)", re.IGNORECASE)
    for p in files:
        m = pat.search(p.stem)
        if not m:
            continue
        key, kind = m.group(1), m.group(2).lower()
        (rings if kind == "rings" else circles)[key] = p
    produced = []
    for key in sorted(set(rings) & set(circles), key=lambda x: int(x)):
        try:
            ring_im = Image.open(rings[key]).convert("RGB")
            circ_im = Image.open(circles[key]).convert("RGB")
        except Exception:
            continue
        for i, (rs, cs) in enumerate(zip(_vsplit4(ring_im), _vsplit4(circ_im))):
            fp = out_dir / f"special_{key}_{i}.png"
            _combine(rs, cs, mode).save(fp); produced.append(fp)
    print(f"[special rotation] {len(set(rings) & set(circles))} matched pairs -> {len(produced)} composites ({mode})")
    return produced

## 3. Gather files per class (existing + new), dedup, stitch

In [ ]:
special_paths = build_special_rotation(PATHS["rot_special"], STITCH_DIR, COMBINE_MODE)

addyrus     = list_images(PATHS["moving_train"])
darkmarket  = list_images(PATHS["moving_test"])
print(f"[moving window] addyrus={len(addyrus)}, darkmarket raw={len(darkmarket)} -> deduping...")
darkmarket_u = dedup_by_pixels(darkmarket)
print(f"[moving window] darkmarket unique={len(darkmarket_u)}")

raw_pools = {
    "Text (Gregwar)":           list_images(PATHS["gregwar"]),
    "Text (Mobicms)":           list_images(PATHS["mobicms"]),
    "Text (King)":              list_images(PATHS["king"]),
    "Text (other)":             [p for r in PATHS["text_other"] for p in list_images(r)],
    "Moving Window":            addyrus + darkmarket_u,
    "Open Circle":              list_images(PATHS["open_circle"]),
    "Image Rotation (normal)":  [p for r in PATHS["rot_normal"] for p in list_images(r)],
    "Image Rotation (special)": special_paths,
    "No Solver":                [p for r in PATHS["no_solver"] for p in list_images(r)],
}

# new classes added in cell 1 are handled here
for name, paths in NEW_PATHS.items():
    raw_pools[name] = [p for r in paths for p in list_images(r)]

print("\nAvailable images per class (before capping):")
for c in CLASSES:
    n = len(raw_pools.get(c, [])); flag = "  <-- EMPTY, check path!" if n == 0 else ""
    print(f"  {c:28s}: {n:>7d}{flag}")

## 4. Cap + stratified per-class split

Per-class split so even the smallest class lands in train/val/test.
**Image Rotation (normal)** uses a group-aware key so a base image's rotated
variants never span splits (a sanity print confirms zero cross-split overlap).

In [ ]:
def split_indices(n, ratios, seed=SEED):
    idx = np.arange(n); rng = np.random.default_rng(seed); rng.shuffle(idx)
    n_tr = int(round(ratios[0] * n)); n_va = int(round(ratios[1] * n))
    return idx[:n_tr], idx[n_tr:n_tr + n_va], idx[n_tr + n_va:]

def normal_group_key(path):
    p = Path(path); stem = p.stem; parent = p.parent.name
    m = re.search(r"iteration_(\d+)_image_(\d+)", stem)
    if m:
        return f"{parent}|it{m.group(1)}|img{m.group(2)}"
    m = re.search(r"iteration_(\d+)", stem)
    if m:
        return f"{parent}|it{m.group(1)}"
    return f"{parent}|{stem}"

def one_per_challenge(paths, key_fn, seed=SEED):
    groups = defaultdict(list)
    for p in paths:
        groups[key_fn(p)].append(p)
    rng = random.Random(seed)
    return [rng.choice(v) for v in groups.values()]

NORMAL = "Image Rotation (normal)"
ONE_PER_CHALLENGE = True

rows = {"train": [], "val": [], "test": []}
rng = np.random.default_rng(SEED)

for c in CLASSES:
    pool = list(raw_pools.get(c, []))
    if len(pool) == 0:
        continue
    if c == NORMAL and ONE_PER_CHALLENGE:
        before = len(pool)
        pool = one_per_challenge(pool, normal_group_key, SEED)
        print(f"[normal] {before} variant images -> {len(pool)} unique challenges (1 variant each)")
    cap = CAP_OVERRIDES.get(c, CAP_PER_CLASS)
    if cap is not None and len(pool) > cap:
        pick = rng.choice(len(pool), size=cap, replace=False)
        pool = [pool[i] for i in pick]
    tr, va, te = split_indices(len(pool), SPLIT)
    for s, ids in zip(["train", "val", "test"], [tr, va, te]):
        rows[s].extend((str(pool[i]), CLASS_TO_IDX[c]) for i in ids)

print("Split sizes:", {s: len(v) for s, v in rows.items()})

def _norm_groups(split):
    return {normal_group_key(p) for p, lbl in rows[split] if lbl == CLASS_TO_IDX[NORMAL]}
gtr, gva, gte = _norm_groups("train"), _norm_groups("val"), _norm_groups("test")
print(f"[normal-rotation challenges] train={len(gtr)} val={len(gva)} test={len(gte)} "
      f"| overlaps tr&va={len(gtr & gva)} tr&te={len(gtr & gte)} va&te={len(gva & gte)}")

import collections
print("\nPer-class train counts (pre-oversample):")
tr_counts = collections.Counter(lbl for _, lbl in rows["train"])
for c in CLASSES:
    print(f"  {c:28s}: {tr_counts.get(CLASS_TO_IDX[c], 0)}")

## 5. Materialise the ImageFolder tree (`train/val/test → <class>/`)

Files are symlinked into the tree Ultralytics expects. Minority train classes are
oversampled with extra symlinks (Ultralytics re-augments each epoch, so repeats
act like a weighted sampler). Val/test are left at the true distribution.

In [ ]:
def link_or_copy(real_src, dst):
    try:
        os.symlink(real_src, dst)
    except FileExistsError:
        pass
    except OSError:
        shutil.copy(real_src, dst)

if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)
for split in ["train", "val", "test"]:
    for c in CLASSES:
        (DATASET_DIR / split / SAFE[c]).mkdir(parents=True, exist_ok=True)

for split in ["train", "val", "test"]:
    counters = defaultdict(int)
    for path, label in rows[split]:
        c = CLASSES[label]; d = DATASET_DIR / split / SAFE[c]
        ext = Path(path).suffix.lower() or ".png"
        link_or_copy(os.path.realpath(path), str(d / f"{counters[c]:06d}{ext}"))
        counters[c] += 1

if OVERSAMPLE_TRAIN:
    train_counts = {c: len(list((DATASET_DIR / "train" / SAFE[c]).iterdir())) for c in CLASSES}
    target = TARGET_TRAIN_PER_CLASS or max(train_counts.values())
    for c in CLASSES:
        d = DATASET_DIR / "train" / SAFE[c]
        files = sorted(d.iterdir()); n = len(files)
        if not files or n >= target:
            continue
        i = 0
        while n < target:
            real = os.path.realpath(files[i % len(files)])
            link_or_copy(real, str(d / f"os_{n:06d}{Path(real).suffix}"))
            n += 1; i += 1
    print("Per-class TRAIN counts (post-oversample, target = %d):" % target)
    for c in CLASSES:
        print(f"  {c:28s}: {len(list((DATASET_DIR / 'train' / SAFE[c]).iterdir()))}")

## 6. Install Ultralytics

In [ ]:
!pip install -q -U ultralytics
import torch
from ultralytics import YOLO
DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

## 7. Train YOLOv8n-cls and save the model

Ultralytics runs its own epoch loop with per-epoch validation and early stopping
(`patience`), saving `best.pt`. The trained model is copied to
`/kaggle/working/best_captcha_classifier.pt`.

In [ ]:
model = YOLO(MODEL)
model.train(
    data=str(DATASET_DIR),
    epochs=EPOCHS, imgsz=IMG_SIZE, batch=BATCH_SIZE,
    patience=PATIENCE, seed=SEED,
    project=str(WORK_DIR / "yolo_runs"),
    name="yolov8n-cls", exist_ok=True, verbose=True,
)

save_dir = Path(model.trainer.save_dir)
best_pt  = getattr(model.trainer, "best", None) or (save_dir / "weights" / "best.pt")

OUT_MODEL = WORK_DIR / "best_captcha_classifier.pt"
shutil.copy(str(best_pt), OUT_MODEL)
print("\nTrained weights:", best_pt)
print("Saved model    :", OUT_MODEL)

## 8. Swap the model into DACCAS (do this yourself)

In [ ]:
# 1) Replace the dispatcher weights:
#       models/CLASSIFICATION_MODEL.pt   <-  best_captcha_classifier.pt
#
# 2) If you ADDED a class, register its safe-name -> readable-name in
#    daccas/classifier.py (SAFE_TO_CLASS), and add the matching solver
#    (BaseSolver subclass + DACCAS.registry entry).
#
# The classifier emits these safe dir-names; copy the new one(s) into SAFE_TO_CLASS:
print("Class label mapping (safe_dir_name -> readable) for this model:")
for safe, readable in SAFE_TO_CLASS.items():
    print(f'    "{safe}": "{readable}",')